In [4]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "ruc_Class25Q2_train_rent.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

txt_col = '客户反馈'
if txt_col not in df.columns:
    raise KeyError(f"找不到列 {txt_col}，请确认。")

# 生成基础文本特征
docs = df[txt_col].fillna("").astype(str).map(normalize_text).tolist()

# 情感分析与句子统计
sent_res = [doc_sentiment_snownlp(t) for t in docs]
df['txt_sent_score'] = [r[0] for r in sent_res]  # 情感得分
df['txt_n_sent_pos'] = [r[1] for r in sent_res]   # 正面句数
df['txt_n_sent_neg'] = [r[2] for r in sent_res]   # 负面句数
df['txt_n_sents'] = [r[3] for r in sent_res]      # 总句数

# 文本长度特征
df['txt_len_chars'] = df[txt_col].fillna("").astype(str).str.len()  # 字符数
df['txt_len_words'] = df[txt_col].fillna("").astype(str).str.count(r'\w+')  # 词数（简单近似）

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
Xred, _, _ = tfidf_svd_fit_transform(docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['txt_sent_score', 'txt_n_sent_pos', 'txt_n_sent_neg', 'txt_n_sents', 'txt_len_chars', 'txt_len_words']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "text_features_with_sentiment_train_rent.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")


正在生成 TF-IDF -> SVD 特征。
数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...
文本特征及情感分析结果已生成并保存为 text_features_with_sentiment_train_rent.csv.


In [21]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "text_features_with_sentiment_train_rent.csv"  # 输入文件名
data = pd.read_csv(file_path)

# 显示数据的前几行和数据基本信息
print(data.head())
print(data.info())

# 1. 处理缺失值
# 检查缺失值
missing_values = data.isnull().sum()
print("缺失值统计：\n", missing_values)

# 对于数值型数据：用均值填补
for column in data.select_dtypes(include=[np.number]).columns:
    data[column].fillna(data[column].mean(), inplace=True)

# 对于文本型数据：用'未知'填补
for column in data.select_dtypes(include=[object]).columns:
    data[column].fillna('未知', inplace=True)

# 对于日期型数据：用最近的有效日期填补
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column].fillna(data[column].max(), inplace=True)

# 2. 删除空白和无效值
data.replace("", np.nan, inplace=True)

# 3. 处理乱码
# 假设如有特定列可能存在编码问题，进行清理：
# data['column_name'] = data['column_name'].str.encode('utf-8').str.decode('utf-8')

# 4. 标准化数据格式
# 标准化日期格式
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')

# 5. 检查并处理异常值
# 这里以价格列为例，进行异常值检测
price_column = "Price"  # 假设你的价格列名为 "price"
q1 = data[price_column].quantile(0.25)
q3 = data[price_column].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

data = data[(data[price_column] >= lower_bound) & (data[price_column] <= upper_bound)]

# 6. 删除重复数据
data.drop_duplicates(inplace=True)

# 7. 数据类型转换
# 确保数值、日期和文本数据的类型正确
for column in data.select_dtypes(include=[object]).columns:
    data[column] = data[column].astype(str)

# 8. 清洗完成后，重新检查数据
print("清洗后的数据:")
print(data.info())
print(data.head())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "cleaned_data1.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


C:\Users\86130\AppData\Local\Temp\ipykernel_16612\3552198298.py:6: DtypeWarning: Columns (23,24,25,26,29,30,32,33,34,35,36,37,38,39,40,42) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path)


   城市      户型   装修          Price      楼层      面积 朝向        交易时间 付款方式 租赁方式  \
0   0  1室1厅1卫  精装修  654646.481811    4/6层  36.42㎡  西  2024-11-28  季付价   整租   
1   0  1室1厅1卫  精装修  665412.057415    4/6层  41.00㎡  南  2024-10-30  季付价   整租   
2   0  1室1厅1卫  精装修  778222.820548   1/18层  37.36㎡  北  2024-11-12  季付价   整租   
3   0  3室1厅2卫  精装修  612084.974699   1/10层  55.42㎡  南  2024-10-14  季付价   整租   
4   0  1室1厅1卫  精装修  994732.124864  18/18层  49.30㎡  南  2024-12-08  季付价   整租   

   ... tfidf_svd_16 tfidf_svd_17 tfidf_svd_18 tfidf_svd_19  \
0  ...    -0.015439     0.029206    -0.021079     0.001527   
1  ...    -0.016708     0.015206    -0.028441    -0.035853   
2  ...     0.045676    -0.040982     0.014745    -0.056928   
3  ...    -0.067201    -0.016207     0.110805     0.042722   
4  ...     0.098828    -0.009056     0.011220    -0.104963   

  group_txt_sent_score group_txt_n_sent_pos group_txt_n_sent_neg  \
0             0.498626             0.437500             0.453125   
1             0.523513

C:\Users\86130\AppData\Local\Temp\ipykernel_16612\3552198298.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[column].fillna(data[column].mean(), inplace=True)
C:\Users\86130\AppData\Local\Temp\ipykernel_16612\3552198298.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


清洗后的数据:
<class 'pandas.core.frame.DataFrame'>
Index: 93365 entries, 0 to 98898
Data columns (total 78 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   城市                    93365 non-null  int64  
 1   户型                    93365 non-null  object 
 2   装修                    93365 non-null  object 
 3   Price                 93365 non-null  float64
 4   楼层                    93365 non-null  object 
 5   面积                    93365 non-null  object 
 6   朝向                    93365 non-null  object 
 7   交易时间                  93365 non-null  object 
 8   付款方式                  93365 non-null  object 
 9   租赁方式                  93365 non-null  object 
 10  电梯                    93365 non-null  object 
 11  车位                    93365 non-null  object 
 12  用水                    93365 non-null  object 
 13  用电                    93365 non-null  object 
 14  燃气                    93365 non-null  object 
 15  采暖              

In [22]:
import pandas as pd

# 读取CSV文件
file_path = 'cleaned_data1.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "户型",
    "装修",
    "楼层",
    "朝向",
    "付款方式",
    "租赁方式",
    "电梯",
    "车位",
    "用水",
    "用电",
    "燃气",
    "采暖",
    "租期",
    "配套设施",
    "环线位置",
    "建筑结构",
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('encod_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'encod_cleaned_data.csv'")

数据量化完成并已保存至 'encod_cleaned_data.csv'


In [23]:
import pandas as pd

# 读取CSV文件
file_path = 'encod_cleaned_data.csv'
data = pd.read_csv(file_path)

# 处理“套内面积”列
def process_area(area):
    if area == '未知' or pd.isna(area) or area == '':
        return 0  # 将“未知”和空字符串替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['面积'] = data['面积'].apply(process_area)

# 打印结果以确认
print(data['面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('processed_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("面积处理完成并已保存至 'processed_cleaned_data.csv'")

0    36.4
1    41.0
2    37.3
3    55.4
4    49.3
Name: 面积, dtype: float64
面积处理完成并已保存至 'processed_cleaned_data.csv'


In [24]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_cleaned_data.csv'
data = pd.read_csv(file_path)

def process_area(area):
    if area == '未知':
        return 0  # 将“未知”替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-1])  # 去掉最后字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['房屋总数'] = data['房屋总数'].apply(process_area)
data['楼栋总数'] = data['楼栋总数'].apply(process_area)
data['绿 化 率'] = data['绿 化 率'].apply(process_area)


# 保存处理后的数据到新的CSV文件
data.to_csv('n.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'n.csv'")

处理完成并已保存至 'n.csv'


In [25]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'n.csv'  
data = pd.read_csv(file_path)

# 处理建筑年代
def process_architecture_years(year_str):
    # 使用正则表达式提取年份
    years = re.findall(r'\d{4}', year_str)
    
    if len(years) == 1:  # 仅有一个年份
        return int(years[0])  # 返回单一年份
    elif len(years) == 2:  # 有两个年份
        return int((int(years[0]) + int(years[1])) / 2)  # 返回中间值
    else:
        return None  # 如果没有年份，返回None

# 应用处理函数
data['建筑年代'] = data['建筑年代'].apply(process_architecture_years)

# 打印结果以确认
print(data[['建筑年代']].head())

# 保存处理后的数据
data.to_csv('a.csv')
print("建筑年代处理完成并已保存至 'a.csv'")

     建筑年代
0  1982.0
1  1995.0
2  2006.0
3  2016.0
4  1988.0
建筑年代处理完成并已保存至 'a.csv'


In [26]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'a.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理物业费
def process_property_fee(fee_str):
    # 检查是否为“未知”
    if fee_str.strip().lower() == "未知":
        return 0  # 将“未知”视为0
    
    # 提取所有的数值 (整数或小数)
    numbers = re.findall(r'\d*\.?\d+', fee_str)  # 提取所有数字
    
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    
    if len(numbers) == 1:  # 仅有一个数值
        return numbers[0]  # 返回该数值
    elif len(numbers) == 2:  # 有两个数值
        return sum(numbers) / 2  # 返回中间值
    else:
        return 0  # 如果无法匹配，有其他情况，返回0

# 应用处理函数
data['物 业 费'] = data['物 业 费'].apply(process_property_fee)
data['燃气费'] = data['燃气费'].apply(process_property_fee)
data['供热费'] = data['供热费'].apply(process_property_fee)

# 保存处理后的数据
data.to_csv('b.csv', index=False, encoding='utf_8_sig')

print("物业费处理完成并已保存至 'b.csv'")


物业费处理完成并已保存至 'b.csv'


In [27]:
import pandas as pd

# 读取CSV文件
file_path = 'b.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "产权描述",
    "供水",
    "供暖",
    "供电",
    "物业类别",   
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('c.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'c.csv'")

数据量化完成并已保存至 'c.csv'


In [28]:
import pandas as pd

# 加载 CSV 文件
df = pd.read_csv('c.csv')  # 替换为你的文件名

# 要删除的列名列表
columns_to_drop = [
    '交易时间',
    '开发商',
    '物业公司',
    '客户反馈',
    '物业办公电话',
    'Unnamed: 0',
    '停车费用'
]

# 删除指定列
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')  # 使用 errors='ignore' 忽略不存在的列

# 验证删除结果
print("删除后的列名:", df.columns)

# 可选：将修改后的 DataFrame 保存到新文件
df.to_csv('d2.csv', index=False)  # 保存为新的 CSV 文件

print("已保存至 'd2.csv'")

删除后的列名: Index(['城市', '户型', '装修', 'Price', '楼层', '面积', '朝向', '付款方式', '租赁方式', '电梯', '车位',
       '用水', '用电', '燃气', '采暖', '租期', '配套设施', 'lon', 'lat', '年份', '区县', '板块',
       '环线位置', '物业类别', '建筑年代', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费',
       '建筑结构', '产权描述', '供水', '供暖', '供电', '燃气费', '供热费', '停车位', 'coord_x',
       'coord_y', 'txt_sent_score', 'txt_n_sent_pos', 'txt_n_sent_neg',
       'txt_n_sents', 'txt_len_chars', 'txt_len_words', 'tfidf_svd_0',
       'tfidf_svd_1', 'tfidf_svd_2', 'tfidf_svd_3', 'tfidf_svd_4',
       'tfidf_svd_5', 'tfidf_svd_6', 'tfidf_svd_7', 'tfidf_svd_8',
       'tfidf_svd_9', 'tfidf_svd_10', 'tfidf_svd_11', 'tfidf_svd_12',
       'tfidf_svd_13', 'tfidf_svd_14', 'tfidf_svd_15', 'tfidf_svd_16',
       'tfidf_svd_17', 'tfidf_svd_18', 'tfidf_svd_19', 'group_txt_sent_score',
       'group_txt_n_sent_pos', 'group_txt_n_sent_neg', 'group_txt_n_sents',
       'group_txt_len_chars', 'group_txt_len_words'],
      dtype='object')
已保存至 'd2.csv'


In [29]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "d2.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的基本信息
print(data.info())

# 假设建筑年代的列名是 "建筑年代"
# 1. 计算均值或中位数
# 这里我们选择使用中位数进行填补
median_year = data['建筑年代'].median()

# 2. 用中位数填补缺失值
data['建筑年代'].fillna(median_year, inplace=True)

# 3. 检查是否填补成功
print("填补后的建筑年代列:")
print(data['建筑年代'].describe())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "e2.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93365 entries, 0 to 93364
Data columns (total 72 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   城市                    93365 non-null  int64  
 1   户型                    93365 non-null  int64  
 2   装修                    93365 non-null  int64  
 3   Price                 93365 non-null  float64
 4   楼层                    93365 non-null  int64  
 5   面积                    93365 non-null  float64
 6   朝向                    93365 non-null  int64  
 7   付款方式                  93365 non-null  int64  
 8   租赁方式                  93365 non-null  int64  
 9   电梯                    93365 non-null  int64  
 10  车位                    93365 non-null  int64  
 11  用水                    93365 non-null  int64  
 12  用电                    93365 non-null  int64  
 13  燃气                    93365 non-null  int64  
 14  采暖                    93365 non-null  int64  
 15  租期                 

C:\Users\86130\AppData\Local\Temp\ipykernel_16612\3149197409.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['建筑年代'].fillna(median_year, inplace=True)


清洗后的数据已保存到 e2.csv


In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

# 尝试导入 SnowNLP 库用于情感分析
try:
    from snownlp import SnowNLP
    _HAVE_SNOW = True
except ImportError:
    _HAVE_SNOW = False

# ---------- 加载数据 ----------
fname = "ruc_Class25Q2_test_rent.csv"  # 数据文件名
if not os.path.exists(fname):
    raise FileNotFoundError(f"{fname} 不存在，请确认文件路径。")

# 尝试读取 CSV 文件，优先使用 UTF-8 编码，如果失败则使用 GBK 编码
try:
    df = pd.read_csv(fname, dtype=str, low_memory=False)
except Exception:
    df = pd.read_csv(fname, dtype=str, encoding='gbk', low_memory=False)

# ---------- 基础函数 ----------
def normalize_text(x):
    """对文本进行规范化处理，包括去除多余的空白字符和特殊字符。"""
    if pd.isna(x): 
        return ""
    s = str(x)
    s = unicodedata.normalize("NFKC", s)  # 将字符规范化
    s = re.sub(r'[\u200B-\u200F\uFEFF]', '', s)  # 去除不可见字符
    return re.sub(r'\s+', ' ', s).strip()  # 简化空格并去掉首尾空格

def doc_sentiment_snownlp(text):
    """计算文本的情感分数及句子统计信息。"""
    if not _HAVE_SNOW:
        return 0.5, 0, 0, 0  # 若未安装 SnowNLP，返回默认值
    s = SnowNLP(text)
    sentiment_score = s.sentiments  # 获取情感得分
    return sentiment_score, int(sentiment_score > 0.6), int(sentiment_score < 0.4), 1  # 返回情感分数和句子统计

# ---------- 特征提取 ----------
def tfidf_svd_fit_transform(texts, max_features=5000, n_components=20, random_state=42):
    """生成TF-IDF特征并进行SVD降维。"""
    tf = TfidfVectorizer(max_features=max_features, token_pattern=r"(?u)\b\w+\b")
    Xtf = tf.fit_transform(texts)  # 进行TF-IDF转换
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)  # SVD降维
    Xred = svd.fit_transform(Xtf)  # 进行SVD变换
    return Xred, tf, svd  # 返回降维后的特征和模型

def oof_group_aggregate(train_df, group_col, feat_cols, n_splits=5, seed=42):
    """使用KFold进行OOF聚合以生成组级文本特征。"""
    oof_dict = {}
    global_means = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for f in feat_cols:
        # 确保每一列只包含数值
        train_df[f] = train_df[f].apply(
            lambda x: x if isinstance(x, (float, int)) else float('nan')  # 非数值数据转为NaN
        )

        # 计算组级特征的均值
        oof = pd.Series(index=train_df.index, dtype=float)
        global_means[f] = train_df[f].mean()  # 全局均值

        for tr, val in kf.split(train_df):
            # 分组计算均值，处理 NaN
            grp_mean = train_df.iloc[tr].groupby(group_col)[f].mean()
            oof.iloc[val] = train_df.iloc[val][group_col].map(grp_mean).fillna(global_means[f])
        
        oof_dict[f] = oof.fillna(global_means[f])  # 若该组无数据，则使用全局均值

    return oof_dict

# ---------- 主流程 ----------
group_col = None
for cand in ['小区', '小区名', '板块', '板块名']:
    if cand in df.columns:
        group_col = cand
        break

if group_col is None:
    raise KeyError("未找到用于聚合的列（如 '小区' 或 '板块'），请确认列名。")

if 'Price' in df.columns:
    try:
        df['Price'] = df['Price'].astype(str).str.replace(',', '').str.replace('元', '').astype(float)
    except Exception:
        df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[^\d\.]', '', regex=True), errors='coerce')

txt_col = '客户反馈'
if txt_col not in df.columns:
    raise KeyError(f"找不到列 {txt_col}，请确认。")

# 生成基础文本特征
docs = df[txt_col].fillna("").astype(str).map(normalize_text).tolist()

# 情感分析与句子统计
sent_res = [doc_sentiment_snownlp(t) for t in docs]
df['txt_sent_score'] = [r[0] for r in sent_res]  # 情感得分
df['txt_n_sent_pos'] = [r[1] for r in sent_res]   # 正面句数
df['txt_n_sent_neg'] = [r[2] for r in sent_res]   # 负面句数
df['txt_n_sents'] = [r[3] for r in sent_res]      # 总句数

# 文本长度特征
df['txt_len_chars'] = df[txt_col].fillna("").astype(str).str.len()  # 字符数
df['txt_len_words'] = df[txt_col].fillna("").astype(str).str.count(r'\w+')  # 词数（简单近似）

# TF-IDF + SVD特征生成
print("正在生成 TF-IDF -> SVD 特征。")
Xred, _, _ = tfidf_svd_fit_transform(docs)  # 生成特征
for i in range(Xred.shape[1]):
    df[f'tfidf_svd_{i}'] = Xred[:, i]  # 添加SVD特征

# ---------- OOF 聚合 ----------
doc_feats = ['txt_sent_score', 'txt_n_sent_pos', 'txt_n_sent_neg', 'txt_n_sents', 'txt_len_chars', 'txt_len_words']

if 'Price' in df.columns and df['Price'].notna().sum() > 0:
    print("数据包含 Price，使用 KFold OOF 生成组级文本特征（CV-safe）...")
    oof_dict = oof_group_aggregate(df, group_col, doc_feats)
    for f, s in oof_dict.items():
        df[f'group_{f}'] = s.values
else:
    print("无 Price，直接用全表 group mean 映射（非 OOF），注意风险。")
    for f in doc_feats:
        mapping = df.groupby(group_col)[f].mean().to_dict()
        df[f'group_{f}'] = df[group_col].map(mapping).fillna(df[f].mean())

# 最终保存结果
output_file = "text_features_with_sentiment_test_rent.csv"  # 输出文件名
df.to_csv(output_file, index=False)  # 将结果保存为 CSV
print(f"文本特征及情感分析结果已生成并保存为 {output_file}.")

In [ ]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "text_features_with_sentiment_test_rent.csv"  # 输入文件名
data = pd.read_csv(file_path)

# 显示数据的前几行和数据基本信息
print(data.head())
print(data.info())

# 1. 处理缺失值
# 检查缺失值
missing_values = data.isnull().sum()
print("缺失值统计：\n", missing_values)

# 对于数值型数据：用均值填补
for column in data.select_dtypes(include=[np.number]).columns:
    data[column].fillna(data[column].mean(), inplace=True)

# 对于文本型数据：用'未知'填补
for column in data.select_dtypes(include=[object]).columns:
    data[column].fillna('未知', inplace=True)

# 对于日期型数据：用最近的有效日期填补
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column].fillna(data[column].max(), inplace=True)

# 2. 删除空白和无效值
data.replace("", np.nan, inplace=True)

# 3. 处理乱码
# 假设如有特定列可能存在编码问题，进行清理：
# data['column_name'] = data['column_name'].str.encode('utf-8').str.decode('utf-8')

# 4. 标准化数据格式
# 标准化日期格式
for column in data.select_dtypes(include=[np.datetime64]).columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')

# 6. 删除重复数据
data.drop_duplicates(inplace=True)

# 7. 数据类型转换
# 确保数值、日期和文本数据的类型正确
for column in data.select_dtypes(include=[object]).columns:
    data[column] = data[column].astype(str)

# 8. 清洗完成后，重新检查数据
print("清洗后的数据:")
print(data.info())
print(data.head())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "cleaned_data1.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")


In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'cleaned_data1.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "户型",
    "装修",
    "楼层",
    "朝向",
    "付款方式",
    "租赁方式",
    "电梯",
    "车位",
    "用水",
    "用电",
    "燃气",
    "采暖",
    "租期",
    "配套设施",
    "环线位置",
    "建筑结构",
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('encod_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'encod_cleaned_data.csv'")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'encod_cleaned_data.csv'
data = pd.read_csv(file_path)

# 处理“套内面积”列
def process_area(area):
    if area == '未知' or pd.isna(area) or area == '':
        return 0  # 将“未知”和空字符串替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-2])  # 去掉最后两个字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['面积'] = data['面积'].apply(process_area)

# 打印结果以确认
print(data['面积'].head())

# 保存处理后的数据到新的CSV文件
data.to_csv('processed_cleaned_data.csv', index=False, encoding='utf_8_sig')

print("面积处理完成并已保存至 'processed_cleaned_data.csv'")

In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'processed_cleaned_data.csv'
data = pd.read_csv(file_path)

def process_area(area):
    if area == '未知':
        return 0  # 将“未知”替换为0
    else:
        # 移除“m²”单位并转为浮点数
        try:
            return float(area[:-1])  # 去掉最后字符
        except ValueError:
            return 0  # 如果转换失败，返回0

# 应用处理函数
data['房屋总数'] = data['房屋总数'].apply(process_area)
data['楼栋总数'] = data['楼栋总数'].apply(process_area)
data['绿 化 率'] = data['绿 化 率'].apply(process_area)


# 保存处理后的数据到新的CSV文件
data.to_csv('n.csv', index=False, encoding='utf_8_sig')

print("处理完成并已保存至 'n.csv'")

In [ ]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'n.csv'  
data = pd.read_csv(file_path)

# 处理建筑年代
def process_architecture_years(year_str):
    # 使用正则表达式提取年份
    years = re.findall(r'\d{4}', year_str)
    
    if len(years) == 1:  # 仅有一个年份
        return int(years[0])  # 返回单一年份
    elif len(years) == 2:  # 有两个年份
        return int((int(years[0]) + int(years[1])) / 2)  # 返回中间值
    else:
        return None  # 如果没有年份，返回None

# 应用处理函数
data['建筑年代'] = data['建筑年代'].apply(process_architecture_years)

# 打印结果以确认
print(data[['建筑年代']].head())

# 保存处理后的数据
data.to_csv('a.csv')
print("建筑年代处理完成并已保存至 'a.csv'")

In [ ]:
import pandas as pd
import re

# 读取CSV文件
file_path = 'a.csv'  # 假设正确的文件路径
data = pd.read_csv(file_path)

# 处理物业费
def process_property_fee(fee_str):
    # 检查是否为“未知”
    if fee_str.strip().lower() == "未知":
        return 0  # 将“未知”视为0
    
    # 提取所有的数值 (整数或小数)
    numbers = re.findall(r'\d*\.?\d+', fee_str)  # 提取所有数字
    
    # 转换为浮点数
    numbers = [float(num) for num in numbers]
    
    if len(numbers) == 1:  # 仅有一个数值
        return numbers[0]  # 返回该数值
    elif len(numbers) == 2:  # 有两个数值
        return sum(numbers) / 2  # 返回中间值
    else:
        return 0  # 如果无法匹配，有其他情况，返回0

# 应用处理函数
data['物 业 费'] = data['物 业 费'].apply(process_property_fee)
data['燃气费'] = data['燃气费'].apply(process_property_fee)
data['供热费'] = data['供热费'].apply(process_property_fee)

# 保存处理后的数据
data.to_csv('b.csv', index=False, encoding='utf_8_sig')

print("物业费处理完成并已保存至 'b.csv'")


In [ ]:
import pandas as pd

# 读取CSV文件
file_path = 'b.csv'
data = pd.read_csv(file_path)

# 需要处理的列名
columns_to_encode = [
    "产权描述",
    "供水",
    "供暖",
    "供电",
    "物业类别",  
    
]

# 标签编码
for column in columns_to_encode:
    if column in data.columns:
        data[column], _ = pd.factorize(data[column])

# 保存到新的CSV文件
data.to_csv('c.csv', index=False, encoding='utf_8_sig')

print("数据量化完成并已保存至 'c.csv'")

In [ ]:
import pandas as pd

# 加载 CSV 文件
df = pd.read_csv('c.csv')  # 替换为你的文件名

# 要删除的列名列表
columns_to_drop = [
    '交易时间',
    '开发商',
    '物业公司',
    '客户反馈',
    '物业办公电话',
    'Unnamed: 0',
    '停车费用',
    'ID'
]

# 删除指定列
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')  # 使用 errors='ignore' 忽略不存在的列

# 验证删除结果
print("删除后的列名:", df.columns)

# 可选：将修改后的 DataFrame 保存到新文件
df.to_csv('d3.csv', index=False)  # 保存为新的 CSV 文件

print("已保存至 'd3.csv'")

In [ ]:
import pandas as pd
import numpy as np

# 加载数据
file_path = "d3.csv"  # 新的输入文件名
data = pd.read_csv(file_path)

# 显示数据的基本信息
print(data.info())

# 假设建筑年代的列名是 "建筑年代"
# 1. 计算均值或中位数
# 这里我们选择使用中位数进行填补
median_year = data['建筑年代'].median()

# 2. 用中位数填补缺失值
data['建筑年代'].fillna(median_year, inplace=True)

# 3. 检查是否填补成功
print("填补后的建筑年代列:")
print(data['建筑年代'].describe())

# 保存清洗后的数据到新的CSV文件
cleaned_file_path = "e3.csv"  # 你可以根据需要修改输出文件名
data.to_csv(cleaned_file_path, index=False)
print(f"清洗后的数据已保存到 {cleaned_file_path}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. 数据加载
print("Loading data...")
data = pd.read_csv('e2.csv')

# 2. 数据处理
print("Processing data...")
if 'community_price' in data.columns:
    data.drop(columns=['community_price'], inplace=True)

# a. 处理缺失值
imputer = SimpleImputer(strategy='mean')
data[:] = imputer.fit_transform(data)

# b. 检测并处理异常值
for column in data.select_dtypes(include=[np.number]).columns:
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data = data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

# 3. 特征和目标分离
X = data.drop(columns=['Price'])
y = data['Price'].clip(lower=0)

# 4. 划分训练集和测试集
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=111)

# 5. 标准化特征
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. 建模
print("Training models...")
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=0.01),
    'Ridge': Ridge(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    # 预测
    y_pred = model.predict(X_test_scaled)
    
    # 修正负值
    y_pred[y_pred < 0] = 0
    y_pred[np.isinf(y_pred)] = 0  # 修正无穷值

    # 计算样本外 MAE
    sample_out_mae = mean_absolute_error(y_test, y_pred)
    results[name] = {
        'Sample Out MAE': sample_out_mae,
        'Train MAE': mean_absolute_error(y_train, model.predict(X_train_scaled)),
        'CV MAE': -cross_val_score(model, X_train_scaled, y_train, cv=6, scoring='neg_mean_absolute_error').mean()
    }

# 7. 汇总结果
output_data = {
    'Metrics': [],
    'In-sample': [],
    'Out-of-sample': [],
    'Cross-validation': [],
    'Kaggle Score': []
}

# 使用示例的Kaggle得分
kaggle_scores = [60, 61, 62, 62]  # 根据具体情况调整

for i, name in enumerate(results.keys()):
    output_data['Metrics'].append(name)
    output_data['In-sample'].append(results[name]['Train MAE'])
    output_data['Out-of-sample'].append(results[name]['Sample Out MAE'])
    output_data['Cross-validation'].append(results[name]['CV MAE'])
    output_data['Kaggle Score'].append(kaggle_scores[i])

# 创建 DataFrame
output_df = pd.DataFrame(output_data)

# 打印结果
print(output_df)

# 8. 保存预测结果
results_df = pd.DataFrame({
    'Actual': y_test,
    'OLS_Predicted': np.clip(models['OLS'].predict(X_test_scaled), 0, None),
    'Lasso_Predicted': np.clip(models['Lasso'].predict(X_test_scaled), 0, None),
    'Ridge_Predicted': np.clip(models['Ridge'].predict(X_test_scaled), 0, None),
    'ElasticNet_Predicted': np.clip(models['ElasticNet'].predict(X_test_scaled), 0, None),
})
results_df.to_csv('result.csv', index=False)
print("预测结果已保存到 'result.csv'")
import joblib
joblib.dump(models['OLS'], 'OLS_model.joblib')
joblib.dump(models['Lasso'], 'Lasso_model.joblib')
joblib.dump(models['Ridge'], 'Ridge_model.joblib')
joblib.dump(models['ElasticNet'], 'ElasticNet_model.joblib')



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

# 1. 加载新数据
print("Loading new data...")
new_data = pd.read_csv('e3.csv')


# 3. 加载之前训练好的模型
print("Loading trained models...")
models = {
    
   'Lasso': joblib.load('Lasso_model.joblib'),
}
# 4. 特征标准化
scaler = StandardScaler()
new_data_scaled = scaler.fit_transform(new_data)

# 5. 进行预测并修正负值
predictions = {}
for name, model in models.items():
    y_pred = model.predict(new_data_scaled)
    y_pred[y_pred < 0] = 0  # 修正负值
    predictions[name] = y_pred

# 6. 创建结果数据框
results_df = pd.DataFrame(predictions)

# 7. 保存预测结果
results_df.to_csv('租金预测.csv', index=False)
print("预测结果已保存")
